# Atmospheric H₂O Trend Detective – Maroc (Google Colab)
**NASA Space Apps Challenge 2026 – Be An Earth System Trend Detective**

Vapeur d'eau atmosphérique (H₂O) mesurée par **NASA Aura MLS** (`ML3MBH2O`, moyennes mensuelles, V005), pour le **Maroc**, de 2004 à 2026, avec l'axe du temps divisé **tous les 6 mois** :
2004 = août → décembre (S2), puis chaque année S1 = janvier → juin et S2 = juillet → décembre.

**Mode d'emploi :** exécutez les cellules **dans l'ordre**, de haut en bas. Chaque étape est dans sa propre cellule ; les 4 graphiques (étapes 5 à 8) se lancent chacun seul une fois les étapes 1 à 4 faites.

| Étape | Rôle |
|---|---|
| 1 | Téléchargement Earthdata (identifiants demandés ici, jamais écrits dans le notebook) |
| 2 | Inspection de la structure d'un fichier |
| 3 | Lecture et nettoyage |
| 4 | Semestres, anomalies, tendances |
| 5–8 | Les 4 graphiques : carte, série temporelle, profil vertical, anomalies |
| 9 | Export CSV |
| 10 | Synthèse : quoi / où / combien / significatif |
| 11 | Données du site web (`donnees.js`) |

> La grille MLS est grossière : sur le Maroc, il n'y a que quelques cellules. Les résultats décrivent la région (Afrique du Nord-Ouest, Atlantique, Méditerranée).

## Préparation

In [ ]:
!pip -q install earthaccess pymannkendall cartopy netCDF4

In [ ]:
# Dossier de travail. UTILISER_DRIVE = True garde données et résultats dans Google Drive
# (sinon tout est perdu à la fermeture de Colab).
UTILISER_DRIVE = False
import os, sys
if UTILISER_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DOSSIER = "/content/drive/MyDrive/h2o_trend_detective"
else:
    DOSSIER = "/content/h2o_trend_detective"
os.makedirs(DOSSIER, exist_ok=True)
os.chdir(DOSSIER)
if DOSSIER not in sys.path:
    sys.path.insert(0, DOSSIER)
print("Dossier de travail :", os.getcwd())

## Fichiers communs
Les trois cellules suivantes écrivent `config.py` (tous les paramètres, dont `ZONE = "Maroc"`), `utils.py` (fonctions communes) et `traces.py` (aides de tracé). Pour changer un paramètre (niveau, zone, seuils), modifiez `config.py` ci-dessous, réexécutez cette cellule, puis lancez `import importlib, config; importlib.reload(config)` et réexécutez les étapes 4 à 11.

In [ ]:
%%writefile config.py
"""Paramètres du projet – tout ce qu'on peut vouloir modifier est ici."""
from pathlib import Path

RACINE = Path(__file__).resolve().parent
DATA_BRUT = RACINE / "data" / "brut"        # fichiers NASA téléchargés
DATA_TRAITE = RACINE / "data" / "traite"    # données intermédiaires
SORTIE = RACINE / "resultats"               # graphiques (PNG) et fichiers CSV
SITE = RACINE / "site"                      # site web statique (HTML/CSS/JS + données prêtes)
for _d in (DATA_BRUT, DATA_TRAITE, SORTIE, SITE / "data" / "csv"):
    _d.mkdir(parents=True, exist_ok=True)

# --- Données NASA Earthdata
PRODUIT = "ML3MBH2O"          # MLS/Aura Level 3 mensuel, vapeur d'eau
VERSION = "005"
DATE_DEBUT = "2004-08-01"
DATE_FIN = "2026-12-31"

# --- Lecture des fichiers (None = choix automatique, voir 02_inspection.py)
GROUPE = None
VARIABLE = None
UNITE = "ppmv"
PLEV_MAX, PLEV_MIN = 316, 0.001   # niveaux (hPa) où le H2O MLS est utilisable

# --- Analyse
NIVEAU = 100.0        # hPa : niveau utilisé pour la carte, la série et les anomalies
ALPHA = 0.05          # seuil de significativité
Z_SEUIL = 1.5         # seuil « semestre exceptionnel » (écarts-types)
FIN_PRE = "2021-12-31"  # fin de la période « avant éruption du Hunga Tonga » (janvier 2022)
TOUS_NIVEAUX = False  # True = CSV de la carte pour tous les niveaux (fichiers très lourds)

# --- Site web : niveaux de pression proposés dans la liste déroulante (le plus proche dans les données est retenu)
NIVEAUX_SITE = [316, 215, 147, 100, 68, 46, 32, 21, 10, 4.6, 1.0, 0.1]

# --- Zone d'étude : "Maroc" ou "Monde" (changer ici, puis relancer les étapes 4 à 11)
ZONE = "Maroc"
# boîte sur laquelle on moyenne (large, car la grille MLS est grossière : quelques cellules seulement sur le Maroc)
BOITES = {"Monde": {}, "Maroc": dict(lat=(24, 38), lon=(-16, 0))}
# fenêtre de la carte : (lon_min, lon_max, lat_min, lat_max) ; None = carte du monde entier
ETENDUES = {"Monde": None, "Maroc": (-20, 5, 20, 42)}

# --- Affichage
AFFICHER = True       # False = enregistrer les images sans ouvrir de fenêtre

# --- Chemins dépendant de la zone
SORTIE_ZONE = SORTIE / ZONE.lower()
SORTIE_ZONE.mkdir(parents=True, exist_ok=True)
PROFIL = DATA_TRAITE / f"profil_{ZONE.lower()}.csv"
PROFIL_PRE = DATA_TRAITE / f"profil_{ZONE.lower()}_pre.csv"


In [ ]:
%%writefile utils.py
"""Fonctions partagées par toutes les étapes (lecture, semestres, statistiques)."""
from types import SimpleNamespace

import netCDF4
import numpy as np
import pandas as pd
import pymannkendall as mk
import xarray as xr
from scipy import stats

import config as cfg


# =====================================================================
# Lecture des fichiers MLS
# =====================================================================
def groupes_3d(g, chemin=""):
    sortie = []
    for n, sg in g.groups.items():
        p = chemin + "/" + n
        if any(len(v.shape) >= 3 for v in sg.variables.values()):
            sortie.append(p)
        sortie += groupes_3d(sg, p)
    return sortie


def chercher(g, mots):
    """Cherche une variable dans le groupe puis dans ses parents."""
    while g is not None:
        for n in g.variables:
            if any(m in n.lower() for m in mots):
                return g.variables[n]
        g = g.parent
    raise KeyError(mots)


def choisir_variable(g):
    cand = [n for n, v in g.variables.items() if len(v.shape) >= 3]
    print("   variables 3D du groupe :", cand)
    pref = [n for n in cand if not any(k in n.lower() for k in ("std", "rms", "min", "max", "prec", "count", "num"))]
    return (pref or cand)[0]


def resoudre_groupe_variable(chemin_fichier):
    """Renvoie (groupe, variable) : valeurs de config.py, sinon choix automatique."""
    with netCDF4.Dataset(chemin_fichier) as f:
        groupe = cfg.GROUPE or groupes_3d(f)[0]
        variable = cfg.VARIABLE or choisir_variable(f[groupe])
    return groupe, variable


def lire_fichier(chemin, debut, groupe, variable):
    """Lit un fichier (peut contenir plusieurs mois) -> tableau (temps, plev, lat, lon) et dates."""
    with netCDF4.Dataset(chemin) as f:
        g = f[groupe]
        v = g.variables[variable]
        a = np.ma.filled(v[:].astype("float64"), np.nan)
        lat = np.array(chercher(g, ["lat"])[:], dtype=float)
        lon = np.array(chercher(g, ["lon"])[:], dtype=float)
        lev = np.array(chercher(g, ["press", "lev"])[:], dtype=float)
        try:
            tv = chercher(g, ["time"])
            tt = [pd.Timestamp(d.year, d.month, 1) for d in
                  netCDF4.num2date(tv[:], tv.units, only_use_cftime_datetimes=False,
                                   only_use_python_datetimes=True)]
        except Exception:
            tt = None
    ax = [list(a.shape).index(len(c)) for c in (lev, lat, lon)]   # axes repérés par leur longueur
    autres = [i for i in range(a.ndim) if i not in ax]
    a = np.transpose(a, ax + autres)
    a = a.reshape(a.shape[:3] + (-1,))      # (plev, lat, lon, temps)
    a = np.moveaxis(a, -1, 0)               # (temps, plev, lat, lon)
    a[np.abs(a) > 1e30] = np.nan
    if tt is None or len(tt) != a.shape[0]:  # repli : mois consécutifs depuis le début du fichier
        tt = list(pd.date_range(debut, periods=a.shape[0], freq="MS"))
    return a, tt, lev, lat, lon


# =====================================================================
# Outils statistiques
# =====================================================================
def moyenne(x, lat=None, lon=None):
    """Moyenne spatiale pondérée par cos(latitude)."""
    if lat:
        x = x.sel(lat=slice(*lat))
    if lon:
        x = x.sel(lon=slice(*lon))
    return x.weighted(np.cos(np.deg2rad(x.lat))).mean(("lat", "lon"))


def temps_annees(idx):
    idx = pd.DatetimeIndex(idx)
    return np.asarray(idx.year + (idx.month - 0.5) / 12, dtype=float)


def regression(x, nmin=36):
    """Pente (par décennie) et p-value en chaque point, corrigée de l'autocorrélation lag-1."""
    t = xr.DataArray(temps_annees(x.time.values), coords={"time": x.time}, dims="time")
    m = x.notnull()
    n = m.sum("time")
    tm, ym = t.where(m).mean("time"), x.mean("time")
    sxx = ((t - tm) ** 2).where(m).sum("time")
    pente = ((t - tm) * (x - ym)).sum("time") / sxx
    res = x - (ym + pente * (t - tm))
    r1 = (res * res.shift(time=1)).sum("time") / (res ** 2).sum("time")
    neff = (n * (1 - r1) / (1 + r1)).clip(min=3, max=None)
    neff = neff.where(neff <= n, n)
    se = np.sqrt((res ** 2).sum("time") / (n - 2) / sxx)
    p = 2 * stats.t.sf(np.abs((pente / se).values), np.maximum(neff.values - 2, 1))
    p = xr.DataArray(p, coords=pente.coords, dims=pente.dims).where(n >= nmin)
    return (pente * 10).where(n >= nmin), p


def analyser_serie(s):
    """s : Series pandas (index = dates). Régression + IC 95 % + Mann-Kendall."""
    s = s.dropna()
    t = temps_annees(s.index)
    r = stats.linregress(t, s.values)
    res = s.values - (r.intercept + r.slope * t)
    r1 = np.corrcoef(res[1:], res[:-1])[0, 1]
    neff = float(np.clip(len(s) * (1 - r1) / (1 + r1), 3, len(s)))
    se = np.sqrt(np.sum(res ** 2) / (len(s) - 2) / np.sum((t - t.mean()) ** 2))
    tc = stats.t.ppf(0.975, max(neff - 2, 1))
    m1, m2 = mk.original_test(s.values), mk.hamed_rao_modification_test(s.values)
    return dict(pente_dec=r.slope * 10, ic_bas=(r.slope - tc * se) * 10, ic_haut=(r.slope + tc * se) * 10,
                p_reg=float(2 * stats.t.sf(abs(r.slope / se), max(neff - 2, 1))),
                mk_p=m1.p, mk_hr_p=m2.p, mk_tendance=m1.trend, n=len(s),
                intercept=r.intercept, slope=r.slope)


# =====================================================================
# Semestres : 2004 = août-décembre ; ensuite S1 = janv-juin, S2 = juil-déc
# =====================================================================
def stats_sem(x):
    """Anomalie et z-score d'un semestre par rapport aux semestres du même type (S1 / S2)."""
    pa, pz = [], []
    for k in (1, 2):
        sub = x.isel(sem=np.where(x.sem.values % 10 == k)[0])
        a = sub - sub.mean("sem")
        pa.append(a)
        pz.append(a / sub.std("sem", ddof=1))
    return xr.concat(pa, "sem").sortby("sem"), xr.concat(pz, "sem").sortby("sem")


def temps_semestres(codes):
    """Date au milieu de chaque semestre (1er avril ou 1er octobre)."""
    return pd.DatetimeIndex([pd.Timestamp(int(c) // 10, 4 if int(c) % 10 == 1 else 10, 1) for c in codes])


def construire_semestres(da):
    """Dataset semestriel : moyennes, anomalies, z-scores, tendances (2 périodes)."""
    sem_np = da.time.dt.year.values * 10 + np.where(da.time.dt.month.values <= 6, 1, 2)
    grp = xr.DataArray(sem_np, dims="time", coords={"time": da.time}, name="sem")
    da_sem = da.groupby(grp).mean("time")
    nb = pd.Series(sem_np).value_counts().sort_index()
    nb_mois = xr.DataArray(nb.values, dims="sem", coords={"sem": nb.index.values})
    anom_sem, z_sem = stats_sem(da_sem)
    anom_t = anom_sem.rename(sem="time").assign_coords(time=temps_semestres(da_sem.sem.values))
    pente, pval = regression(anom_t, nmin=20)
    pente_pre, pval_pre = regression(anom_t.sel(time=slice(None, cfg.FIN_PRE)), nmin=20)
    return xr.Dataset({"h2o_sem": da_sem, "anom_sem": anom_sem, "z_sem": z_sem, "nb_mois": nb_mois,
                       "h2o_moyen": da.mean("time"),
                       "pente": pente, "pval": pval, "pente_pre": pente_pre, "pval_pre": pval_pre})


def charger_contexte():
    """Relit data/traite/semestres.nc et prépare tout ce qu'il faut pour les graphiques."""
    ds = xr.open_dataset(cfg.DATA_TRAITE / "semestres.nc").load()
    codes = ds.sem.values.astype(int)
    etiq = [f"{c // 10}-S{c % 10}" for c in codes]
    t_sem = temps_semestres(codes)
    niv = float(ds.plev.sel(plev=cfg.NIVEAU, method="nearest"))
    m = moyenne(ds["h2o_sem"], **cfg.BOITES[cfg.ZONE])   # (sem, plev) : moyenne sur la zone
    a, z = stats_sem(m)
    return SimpleNamespace(
        zone=cfg.ZONE, ds=ds, codes=codes, etiq=etiq, t_sem=t_sem, n_sem=len(codes), niv=niv,
        m=m, a=a, z=z,
        serie_sem=pd.Series(m.sel(plev=niv).values, index=t_sem),
        serie_anom=pd.Series(a.sel(plev=niv).values, index=t_sem),
        nb_mois=pd.Series(ds["nb_mois"].values, index=codes),
        hunga=etiq.index("2022-S1") if "2022-S1" in etiq else None,
    )


def tableau_profil(ctx, fin=None):
    """Tendance du H2O mondial à chaque niveau de pression (profil vertical complet)."""
    lignes = []
    periode = f"2004–{fin[:4]}" if fin else f"2004–{ctx.etiq[-1][:4]}"
    for p in ctx.m.plev.values:
        s = pd.Series(ctx.a.sel(plev=p).values, index=ctx.t_sem)
        if fin:
            s = s[s.index <= pd.Timestamp(fin)]
        if s.notna().sum() < 20:
            continue
        r = analyser_serie(s)
        moy = float(ctx.m.sel(plev=p).mean())
        lignes.append(dict(zone=cfg.ZONE, periode=periode, plev_hPa=round(float(p), 3), h2o_moyen_ppmv=moy,
                           pente_ppmv_par_decennie=r["pente_dec"], ic95_bas=r["ic_bas"], ic95_haut=r["ic_haut"],
                           pct_par_decennie=r["pente_dec"] / moy * 100,
                           p_regression=r["p_reg"], p_mann_kendall=r["mk_p"],
                           p_mann_kendall_hamed_rao=r["mk_hr_p"],
                           significatif=bool(r["p_reg"] < cfg.ALPHA and r["mk_hr_p"] < cfg.ALPHA)))
    return pd.DataFrame(lignes)


def zoom(x):
    """Restreint un DataArray (lat, lon) à la fenêtre de la zone ; inchangé pour le monde entier."""
    e = cfg.ETENDUES[cfg.ZONE]
    return x if e is None else x.sel(lat=slice(e[2], e[3]), lon=slice(e[0], e[1]))


def texte_zone():
    b = cfg.BOITES[cfg.ZONE]
    if not b:
        return "le monde entier"
    (la0, la1), (lo0, lo1) = b["lat"], b["lon"]
    f = lambda v: f"{abs(v)}°{'O' if v < 0 else 'E'}"
    return f"{cfg.ZONE} (boîte {la0}–{la1}°N, {f(lo0)}–{f(lo1)})"


def vers_df(champs):
    """Dict {nom: DataArray} -> DataFrame (une ligne par point de grille)."""
    df = xr.Dataset(champs).reset_coords(drop=True).to_dataframe().reset_index()
    return df.dropna(subset=[list(champs)[0]]).rename(columns={"plev": "plev_hPa"})


In [ ]:
%%writefile traces.py
"""Petites fonctions de tracé communes aux graphiques."""
import numpy as np


def axe_semestres(ax, ctx):
    x = np.arange(ctx.n_sem)
    ax.set_xticks(x)
    ax.set_xticklabels(ctx.etiq, rotation=90, fontsize=7)
    ax.set_xlim(-.5, ctx.n_sem - .5)


def ligne_hunga(ax, ctx):
    """Repère l'éruption du Hunga Tonga (janvier 2022)."""
    if ctx.hunga is not None:
        ax.axvline(ctx.hunga, color="darkorange", ls="--", lw=1.3)
        ax.text(ctx.hunga, .98, " Hunga Tonga", color="darkorange", fontsize=8,
                va="top", transform=ax.get_xaxis_transform())


## Étape 1 – Téléchargement Earthdata
Compte gratuit : https://urs.earthdata.nasa.gov. Les identifiants sont **saisis au clavier quand la cellule les demande** : ne les écrivez jamais dans le notebook.

Option plus pratique : les **Secrets** de Colab (icône clé à gauche), nommés `EARTHDATA_USERNAME` et `EARTHDATA_PASSWORD`. Dans ce cas, exécutez d'abord :
```python
import os
from google.colab import userdata
os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
```

In [ ]:
"""ÉTAPE 1 – Téléchargement des fichiers NASA Aura MLS (H2O) depuis Earthdata.

Prérequis : un compte gratuit https://urs.earthdata.nasa.gov
Identifiants : variables d'environnement EARTHDATA_USERNAME / EARTHDATA_PASSWORD,
ou fichier ~/.netrc, ou saisie au clavier (la première fois).
Produit : data/brut/*.nc4  et  data/brut/index.csv (fichier + date de début).
"""
import earthaccess
import pandas as pd

import config as cfg


def main():
    earthaccess.login(persist=True)
    resultats = earthaccess.search_data(short_name=cfg.PRODUIT, version=cfg.VERSION,
                                        temporal=(cfg.DATE_DEBUT, cfg.DATE_FIN))
    print(len(resultats), "fichiers trouvés")
    earthaccess.download(resultats, str(cfg.DATA_BRUT))

    lignes = []
    for r in resultats:
        liens = r.data_links()
        liens_donnees = [l for l in liens if l.lower().endswith((".nc", ".nc4", ".he5", ".h5"))]
        nom = (liens_donnees or liens)[0].split("/")[-1]
        debut = pd.Timestamp(r["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"])
        debut = debut.tz_convert(None) if debut.tzinfo else debut
        lignes.append({"fichier": nom, "debut": debut.to_period("M").to_timestamp()})
    index = pd.DataFrame(lignes).sort_values("debut")
    index.to_csv(cfg.DATA_BRUT / "index.csv", index=False)
    print(f"Index enregistré : {cfg.DATA_BRUT / 'index.csv'}")
    print(index.iloc[[0, -1]].to_string(index=False))


main()

## Étape 2 – Inspection de la structure d'un fichier
Affiche les groupes et variables du premier fichier, puis le groupe et la variable utilisés. Si le choix automatique ne convient pas, renseignez `GROUPE` et `VARIABLE` dans `config.py`.

In [ ]:
"""ÉTAPE 2 – Inspection de la structure d'un fichier MLS.

Affiche les groupes et variables du premier fichier téléchargé, puis le groupe et la
variable qui seront utilisés. Si le choix automatique ne convient pas, renseignez
GROUPE et VARIABLE dans config.py.
"""
import netCDF4
import pandas as pd

import config as cfg
import utils


def parcourir(g, pref=""):
    for n, v in g.variables.items():
        print(f"{pref}{n:28s} {v.dimensions} {v.shape}")
    for n, sg in g.groups.items():
        print(f"{pref}[groupe] {n}")
        parcourir(sg, pref + "    ")


def main():
    index = pd.read_csv(cfg.DATA_BRUT / "index.csv")
    premier = cfg.DATA_BRUT / index.fichier.iloc[0]
    print("Fichier :", premier.name, "\n")
    with netCDF4.Dataset(premier) as f:
        parcourir(f)
    groupe, variable = utils.resoudre_groupe_variable(premier)
    print(f"\n>>> Groupe utilisé : {groupe} | Variable utilisée : {variable}")


main()

## Étape 3 – Lecture et nettoyage
La ligne « Mois lus » doit afficher environ 270 mois.

In [ ]:
"""ÉTAPE 3 – Lecture de tous les fichiers, nettoyage, unités.

Produit : data/traite/h2o_mensuel.nc  (H2O mensuel : temps, plev, lat, lon, en ppmv).
"""
import numpy as np
import pandas as pd
import xarray as xr

import config as cfg
import utils


def main():
    index = pd.read_csv(cfg.DATA_BRUT / "index.csv", parse_dates=["debut"])
    chemins = [cfg.DATA_BRUT / f for f in index.fichier]
    groupe, variable = utils.resoudre_groupe_variable(chemins[0])
    print("Groupe :", groupe, "| Variable :", variable)

    piles, temps, lev, lat, lon = [], [], None, None, None
    for chemin, debut in zip(chemins, index.debut):
        a, tt, lev, lat, lon = utils.lire_fichier(chemin, debut, groupe, variable)
        piles.append(a)
        temps += tt
    print("Mois lus :", len(temps), "| de", str(temps[0])[:7], "à", str(temps[-1])[:7])

    da = xr.DataArray(np.concatenate(piles), dims=("time", "plev", "lat", "lon"),
                      coords={"time": temps, "plev": lev, "lat": lat, "lon": lon}, name="H2O")
    da = da.sortby("time")
    da = da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby("lon").sortby("lat")
    da = da.sortby("plev", ascending=False)           # de la basse vers la haute atmosphère
    assert da.sizes["time"] > 200, "Trop peu de mois : vérifier l'axe du temps (02_inspection.py)"

    # --- nettoyage
    if float(da.median()) < 1e-3:                     # fraction volumique -> ppmv
        da = da * 1e6
    da = da.sel(plev=slice(cfg.PLEV_MAX, cfg.PLEV_MIN))
    da.attrs["unite"] = cfg.UNITE

    print("Niveaux (hPa) :", np.round(da.plev.values, 3))
    print("Valeurs manquantes (%) :", round(float(da.isnull().mean()) * 100, 1))
    print("Médiane :", round(float(da.median()), 2), cfg.UNITE)

    sortie = cfg.DATA_TRAITE / "h2o_mensuel.nc"
    da.to_netcdf(sortie)
    print("Enregistré :", sortie)


main()

## Étape 4 – Semestres, anomalies, tendances
Tendances calculées sur 2004–2026 **et** sur 2004–2021 (avant l'éruption du Hunga Tonga, janvier 2022).

In [ ]:
"""ÉTAPE 4 – Passage aux semestres, anomalies, tendances.

Découpage : 2004 = août -> décembre (S2), puis pour chaque année S1 = janvier -> juin
et S2 = juillet -> décembre. Anomalie / z-score = écart aux semestres du même type.
Tendances calculées sur 2004-2026 ET sur 2004-2021 (avant l'éruption du Hunga Tonga).

Produit : data/traite/semestres.nc, profil_<zone>.csv, profil_<zone>_pre.csv (zone choisie dans config.py)
"""
import xarray as xr

import config as cfg
import utils


def main():
    da = xr.open_dataarray(cfg.DATA_TRAITE / "h2o_mensuel.nc").load()
    ds = utils.construire_semestres(da)
    ds.to_netcdf(cfg.DATA_TRAITE / "semestres.nc")

    ctx = utils.charger_contexte()
    print("Zone :", ctx.zone)
    print(ctx.n_sem, "semestres :", ctx.etiq[0], "->", ctx.etiq[-1])
    incomplets = [e for e, c in zip(ctx.etiq, ctx.codes) if ctx.nb_mois[c] != (5 if e == "2004-S2" else 6)]
    if incomplets:
        print("Semestres incomplets :", incomplets)

    utils.tableau_profil(ctx).to_csv(cfg.PROFIL, index=False)
    utils.tableau_profil(ctx, cfg.FIN_PRE).to_csv(cfg.PROFIL_PRE, index=False)
    print("Enregistré dans", cfg.DATA_TRAITE)


main()

## Étape 5 – Graphique 1 : carte du Maroc (tendance)
Points noirs = tendance significative ; cadre noir = zone moyennée.

In [ ]:
"""ÉTAPE 5 – Graphique 1 : carte de la tendance du H2O (points = significatif).
Zone « Maroc » : carte zoomée sur le Maroc avec le cadre de la zone moyennée.
Zone « Monde » : carte du monde entier."""
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np

import config as cfg
import utils


def main():
    ctx = utils.charger_contexte()
    pn = utils.zoom(ctx.ds.pente.sel(plev=ctx.niv))
    sn = utils.zoom(ctx.ds.pval.sel(plev=ctx.niv)) < cfg.ALPHA
    lim = float(np.nanpercentile(np.abs(pn), 95))
    if not np.isfinite(lim) or lim == 0:
        lim = 1.0
    e = cfg.ETENDUES[ctx.zone]

    fig = plt.figure(figsize=(11, 5.5) if e is None else (9, 6.5))
    if e is None:
        ax = plt.axes(projection=ccrs.Robinson())
        ax.set_global()
    else:
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent(e)
        ax.gridlines(draw_labels=True, lw=.2)
    im = ax.pcolormesh(pn.lon, pn.lat, pn, transform=ccrs.PlateCarree(), cmap="RdBu_r",
                       vmin=-lim, vmax=lim, shading="auto")
    LON, LAT = np.meshgrid(sn.lon, sn.lat)
    ax.scatter(LON[sn.values], LAT[sn.values], s=18 if e else 5, c="k", transform=ccrs.PlateCarree(),
               label=f"significatif (p < {cfg.ALPHA})")
    if e:
        b = cfg.BOITES[ctx.zone]
        ax.add_patch(plt.Rectangle((b["lon"][0], b["lat"][0]), b["lon"][1] - b["lon"][0], b["lat"][1] - b["lat"][0],
                                   fill=False, ec="k", lw=1.5, transform=ccrs.PlateCarree(), label="zone moyennée"))
    ax.coastlines(lw=.8)
    ax.add_feature(cfeature.BORDERS, lw=.4)
    plt.colorbar(im, ax=ax, shrink=.7, label=f"{cfg.UNITE} par décennie")
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title(f"1. Carte {ctx.zone} – tendance du H₂O à {ctx.niv} hPa, 2004–{ctx.etiq[-1][:4]} (semestres)", fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "1_carte.png", dpi=170, bbox_inches="tight")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 6 – Graphique 2 : série temporelle (un point tous les 6 mois)

In [ ]:
"""ÉTAPE 6 – Graphique 2 : série temporelle de la zone (un point tous les 6 mois) et tendance."""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config as cfg
import traces
import utils


def main():
    ctx = utils.charger_contexte()
    y = ctx.serie_anom
    r = utils.analyser_serie(y)
    r2 = utils.analyser_serie(y[y.index <= cfg.FIN_PRE])
    x = np.arange(ctx.n_sem)
    t = utils.temps_annees(ctx.t_sem)
    ybar, tbar = y.mean(), t.mean()
    npre = int((ctx.t_sem <= pd.Timestamp(cfg.FIN_PRE)).sum())
    fin = ctx.etiq[-1][:4]

    fig, ax = plt.subplots(figsize=(14, 5.5))
    ax.fill_between(x, ybar + r["ic_bas"] / 10 * (t - tbar), ybar + r["ic_haut"] / 10 * (t - tbar),
                    color="#9ec5ee", alpha=.5, label=f"IC 95 % (2004–{fin})")
    ax.plot(x, y.values, color="grey", lw=1, marker="o", ms=4, label="Anomalie semestrielle")
    ax.plot(x, r["intercept"] + r["slope"] * t, color="navy", lw=2, label=f"Tendance 2004–{fin}")
    ax.plot(x[:npre], r2["intercept"] + r2["slope"] * t[:npre], color="darkorange", lw=2, ls="--",
            label="Tendance 2004–2021 (avant Hunga Tonga)")
    traces.ligne_hunga(ax, ctx)
    ax.axhline(0, color="k", lw=.5)
    ax.grid(alpha=.3)
    traces.axe_semestres(ax, ctx)
    ax.set_ylabel(f"Anomalie H₂O ({cfg.UNITE})")
    ax.legend(fontsize=8, loc="upper left")
    ax.text(.99, .03,
            f"2004–{fin} : {r['pente_dec']:+.3f} [{r['ic_bas']:+.3f} ; {r['ic_haut']:+.3f}] {cfg.UNITE}/décennie, "
            f"p = {r['p_reg']:.3f}, MK corrigé p = {r['mk_hr_p']:.3f}\n"
            f"2004–2021 : {r2['pente_dec']:+.3f} [{r2['ic_bas']:+.3f} ; {r2['ic_haut']:+.3f}] {cfg.UNITE}/décennie, "
            f"p = {r2['p_reg']:.3f}, MK corrigé p = {r2['mk_hr_p']:.3f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8, bbox=dict(fc="white", alpha=.85))
    ax.set_title(f"2. Graphe temporel – H₂O {ctx.zone} à {ctx.niv} hPa, 2004 à {fin}, un point tous les 6 mois", fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "2_graphe_temporel.png", dpi=170, bbox_inches="tight")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 7 – Graphique 3 : profil vertical des tendances

In [ ]:
"""ÉTAPE 7 – Graphique 3 : profil vertical des tendances du H2O (zone choisie), deux périodes."""
import matplotlib.pyplot as plt
import pandas as pd

import config as cfg
import utils


def main():
    ctx = utils.charger_contexte()
    tab = pd.read_csv(cfg.PROFIL)
    tab_pre = pd.read_csv(cfg.PROFIL_PRE)

    fig, ax = plt.subplots(figsize=(7, 6.5))
    for t, coul, ls, lab in [(tab, "navy", "-", f"2004–{ctx.etiq[-1][:4]}"),
                             (tab_pre, "darkorange", "--", "2004–2021 (avant Hunga Tonga)")]:
        ax.fill_betweenx(t.plev_hPa, t.ic95_bas, t.ic95_haut, color=coul, alpha=.12)
        ax.plot(t.pente_ppmv_par_decennie, t.plev_hPa, "o", ls=ls, color=coul, ms=4, label=lab)
        s = t.significatif.astype(bool)
        ax.scatter(t.pente_ppmv_par_decennie[s], t.plev_hPa[s], color="red", s=24, zorder=5)
    ax.axvline(0, color="k", lw=.5)
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.grid(alpha=.3)
    ax.legend(fontsize=8, title="Période (rouge = significatif)")
    ax.set_xlabel(f"Tendance ({cfg.UNITE} par décennie)")
    ax.set_ylabel("Pression (hPa)")
    ax.set_title(f"3. Profil vertical des tendances du H₂O – {ctx.zone}", fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "3_profil_vertical.png", dpi=170, bbox_inches="tight")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 8 – Graphique 4 : détection d'anomalies
Les semestres exceptionnellement élevés sont en rouge.

In [ ]:
"""ÉTAPE 8 – Graphique 4 : détection d'anomalies (semestres exceptionnellement élevés)."""
import matplotlib.pyplot as plt
import numpy as np

import config as cfg
import traces
import utils


def main():
    ctx = utils.charger_contexte()
    an = ctx.a.sel(plev=ctx.niv).values
    zn = ctx.z.sel(plev=ctx.niv).values
    sd = {k: float(np.nanstd(an[ctx.codes % 10 == k], ddof=1)) for k in (1, 2)}
    bande = np.array([cfg.Z_SEUIL * sd[c % 10] for c in ctx.codes])
    haut = zn >= cfg.Z_SEUIL
    x = np.arange(ctx.n_sem)

    fig, ax = plt.subplots(figsize=(14, 5.5))
    ax.fill_between(x, -bande, bande, color="grey", alpha=.25, label=f"Normal (±{cfg.Z_SEUIL} σ)")
    ax.axhline(0, color="k", lw=.6)
    ax.plot(x, an, color="#2b3a4a", lw=1.5, marker="o", ms=3)
    ax.scatter(x[haut], an[haut], color="red", s=60, zorder=5, label="Semestre exceptionnellement élevé")
    for xi, yi, lab in zip(x[haut], an[haut], np.array(ctx.etiq)[haut]):
        ax.annotate(lab, (xi, yi), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="red")
    traces.ligne_hunga(ax, ctx)
    traces.axe_semestres(ax, ctx)
    ax.grid(alpha=.3)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_ylabel(f"Anomalie H₂O ({cfg.UNITE})")
    ax.set_title(f"4. Détection d'anomalies – H₂O {ctx.zone} à {ctx.niv} hPa (semestres exceptionnellement élevés)", fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "4_detection_anomalies.png", dpi=170, bbox_inches="tight")
    print("Semestres exceptionnellement élevés :", list(np.array(ctx.etiq)[haut]) or "aucun")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 9 – Export CSV
Un CSV par graphique dans `resultats/maroc/`.

In [ ]:
"""ÉTAPE 9 – Export des résultats en CSV (un fichier par graphique) + archive ZIP.

Produit dans resultats/<zone>/ : 1_carte.csv, 2_graphe_temporel.csv, 3_profil_vertical.csv,
4_detection_anomalies.csv, resultats_H2O_<zone>.zip
"""
import zipfile

import pandas as pd

import config as cfg
import utils


def main():
    ctx = utils.charger_contexte()
    ds, sortie = ctx.ds, cfg.SORTIE_ZONE
    niveaux = [float(p) for p in ds.plev.values] if cfg.TOUS_NIVEAUX else [ctx.niv]
    enc = dict(index=False, encoding="utf-8-sig")

    # 1. Carte (fenêtre de la zone)
    champs = {"h2o_moyen_ppmv": ds.h2o_moyen.sel(plev=niveaux),
              "tendance_ppmv_par_decennie": ds.pente.sel(plev=niveaux), "p_value": ds.pval.sel(plev=niveaux),
              "tendance_2004_2021_ppmv_par_decennie": ds.pente_pre.sel(plev=niveaux),
              "p_value_2004_2021": ds.pval_pre.sel(plev=niveaux)}
    df1 = utils.vers_df({k: utils.zoom(v) for k, v in champs.items()})
    df1["significatif"] = df1.p_value < cfg.ALPHA
    df1["significatif_2004_2021"] = df1.p_value_2004_2021 < cfg.ALPHA
    b = cfg.BOITES[ctx.zone]
    if b:
        df1["dans_zone_moyennee"] = df1.lat.between(*b["lat"]) & df1.lon.between(*b["lon"])
    df1.round(5).to_csv(sortie / "1_carte.csv", **enc)

    # 2. Graphe temporel
    y = ctx.serie_anom
    r = utils.analyser_serie(y)
    r2 = utils.analyser_serie(y[y.index <= cfg.FIN_PRE])
    t = utils.temps_annees(ctx.t_sem)
    lignes = []
    for i, c in enumerate(ctx.codes):
        lignes.append(dict(
            semestre=ctx.etiq[i], date_milieu=ctx.t_sem[i].strftime("%Y-%m-%d"), zone=ctx.zone, plev_hPa=ctx.niv,
            nb_mois=int(ctx.nb_mois[c]), h2o_moyen_ppmv=ctx.serie_sem.iloc[i], anomalie_ppmv=y.iloc[i],
            ligne_tendance_ppmv=r["intercept"] + r["slope"] * t[i],
            pente_ppmv_par_decennie=r["pente_dec"], ic95_bas=r["ic_bas"], ic95_haut=r["ic_haut"],
            p_regression=r["p_reg"], p_mann_kendall=r["mk_p"], p_mann_kendall_hamed_rao=r["mk_hr_p"],
            tendance=r["mk_tendance"],
            pente_2004_2021_ppmv_par_decennie=r2["pente_dec"], ic95_bas_2004_2021=r2["ic_bas"],
            ic95_haut_2004_2021=r2["ic_haut"], p_regression_2004_2021=r2["p_reg"],
            p_mann_kendall_hamed_rao_2004_2021=r2["mk_hr_p"]))
    pd.DataFrame(lignes).round(5).to_csv(sortie / "2_graphe_temporel.csv", **enc)

    # 3. Profil vertical (deux périodes)
    prof = pd.concat([pd.read_csv(cfg.PROFIL), pd.read_csv(cfg.PROFIL_PRE)])
    prof.round(5).to_csv(sortie / "3_profil_vertical.csv", **enc)

    # 4. Détection d'anomalies (tous les niveaux)
    blocs = []
    for i, c in enumerate(ctx.codes):
        d = pd.DataFrame({"semestre": ctx.etiq[i], "zone": ctx.zone, "plev_hPa": ctx.m.plev.values,
                          "h2o_moyen_ppmv": ctx.m.values[i], "anomalie_ppmv": ctx.a.values[i],
                          "z_score": ctx.z.values[i]})
        d["nb_mois"] = int(ctx.nb_mois[c])
        blocs.append(d)
    df4 = pd.concat(blocs)
    df4["exceptionnellement_eleve"] = df4.z_score >= cfg.Z_SEUIL
    df4["exceptionnellement_bas"] = df4.z_score <= -cfg.Z_SEUIL
    df4.round(4).to_csv(sortie / "4_detection_anomalies.csv", **enc)

    # Archive ZIP
    with zipfile.ZipFile(sortie / f"resultats_H2O_{ctx.zone.lower()}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(sortie.iterdir()):
            if f.suffix in (".csv", ".png"):
                zf.write(f, f.name)
    print("Fichiers écrits dans", sortie)


main()

## Étape 10 – Synthèse : quoi / où / combien / significatif

In [ ]:
"""ÉTAPE 10 – Synthèse : Quoi ? Où ? Combien ? Est-ce significatif ?"""
import config as cfg
import utils


def main():
    ctx = utils.charger_contexte()
    y = ctx.serie_anom
    resultats = [(f"2004–{ctx.etiq[-1][:4]}", utils.analyser_serie(y)),
                 ("2004–2021 (avant Hunga Tonga)", utils.analyser_serie(y[y.index <= cfg.FIN_PRE]))]
    print(f"QUOI  : vapeur d'eau atmosphérique (H₂O, Aura MLS), anomalies semestrielles – {utils.texte_zone()}")
    print("OÙ    : carte des tendances et profil vertical")
    print("COMBIEN / SIGNIFICATIF :")
    for lib, r in resultats:
        sens = "augmente" if r["pente_dec"] > 0 else "diminue"
        sig = "SIGNIFICATIVE" if (r["p_reg"] < cfg.ALPHA and r["mk_hr_p"] < cfg.ALPHA) else "non significative"
        print(f"  - {lib}, {ctx.niv} hPa : le H₂O {sens} de {r['pente_dec']:+.3f} {cfg.UNITE}/décennie "
              f"(IC 95 % [{r['ic_bas']:+.3f} ; {r['ic_haut']:+.3f}]) – {sig} "
              f"(p = {r['p_reg']:.3f}, Mann-Kendall corrigé p = {r['mk_hr_p']:.3f}).")


main()

## Étape 11 – Données du site web
Crée `site/data/donnees.js` (résultats déjà calculés : le site n'a besoin d'aucune authentification NASA).

In [ ]:
"""ÉTAPE 11 – Prépare les données du site web (aucune authentification NASA n'est nécessaire sur le site).

Le site est statique : il lit uniquement le fichier site/data/donnees.js, généré ici à partir
des résultats DÉJÀ calculés (data/traite/semestres.nc). Il ne contient ni identifiant ni mot de passe,
ni aucun appel vers Earthdata. Le téléchargement NASA (étape 1) se fait uniquement sur votre ordinateur.

Produit : site/data/donnees.js  et  site/data/csv/*.csv (copie des CSV de l'étape 9, zone choisie dans config.py).
"""
import json
import shutil
from datetime import date

import numpy as np
import pandas as pd

import config as cfg
import utils


def liste(a, n=3):
    """Tableau -> liste JSON ; NaN -> null."""
    a = np.asarray(a, dtype=float).ravel()
    return [round(float(v), n) if np.isfinite(v) else None for v in a]


def stat(r):
    return dict(pente=float(r["pente_dec"]), ic_bas=float(r["ic_bas"]), ic_haut=float(r["ic_haut"]),
                p_reg=float(r["p_reg"]), mk_p=float(r["mk_p"]), mk_hr_p=float(r["mk_hr_p"]),
                tendance=str(r["mk_tendance"]), n=int(r["n"]))


def main():
    ctx = utils.charger_contexte()
    ds = ctx.ds
    dispo = ds.plev.values
    voulus = list(cfg.NIVEAUX_SITE) + [ctx.niv]
    niveaux = sorted({float(dispo[np.abs(np.log(dispo) - np.log(n)).argmin()]) for n in voulus}, reverse=True)

    t = utils.temps_annees(ctx.t_sem)
    npre = int((ctx.t_sem <= pd.Timestamp(cfg.FIN_PRE)).sum())
    sorties = []
    for p in niveaux:
        a_s = pd.Series(ctx.a.sel(plev=p).values, index=ctx.t_sem)
        r = utils.analyser_serie(a_s)
        r2 = utils.analyser_serie(a_s[a_s.index <= cfg.FIN_PRE])
        sd = {k: float(np.nanstd(a_s.values[ctx.codes % 10 == k], ddof=1)) for k in (1, 2)}
        bande = [cfg.Z_SEUIL * sd[int(c) % 10] for c in ctx.codes]
        ybar, tbar = float(a_s.mean()), float(t.mean())
        pente, pval = utils.zoom(ds.pente.sel(plev=p)), utils.zoom(ds.pval.sel(plev=p))
        pente_pre, pval_pre = utils.zoom(ds.pente_pre.sel(plev=p)), utils.zoom(ds.pval_pre.sel(plev=p))
        sorties.append(dict(
            plev=p,
            moyenne=liste(ctx.m.sel(plev=p).values, 4),
            anomalie=liste(a_s.values, 4),
            z=liste(ctx.z.sel(plev=p).values, 3),
            bande=liste(bande, 4),
            tendance=stat(r), tendance_pre=stat(r2),
            ligne=liste(r["intercept"] + r["slope"] * t, 4),
            ligne_pre=liste(r2["intercept"] + r2["slope"] * t[:npre], 4),
            ic_bas_ligne=liste(ybar + r["ic_bas"] / 10 * (t - tbar), 4),
            ic_haut_ligne=liste(ybar + r["ic_haut"] / 10 * (t - tbar), 4),
            carte=dict(pente=liste(pente.values, 4), sig=[int(v) for v in (pval.values < cfg.ALPHA).ravel()],
                       pente_pre=liste(pente_pre.values, 4), sig_pre=[int(v) for v in (pval_pre.values < cfg.ALPHA).ravel()]),
        ))

    profils = []
    for f in (cfg.PROFIL, cfg.PROFIL_PRE):
        df = pd.read_csv(f)
        profils += json.loads(df.round(5).to_json(orient="records"))

    carte_ref = utils.zoom(ds.pente.isel(plev=0))
    e = cfg.ETENDUES[cfg.ZONE]
    dlon = float(np.median(np.diff(carte_ref.lon.values)))
    taille = 8 if e is None else int(min(120, max(8, 40 * dlon)))   # taille des carrés de la carte (px)
    donnees = dict(
        meta=dict(zone=cfg.ZONE, zone_texte=utils.texte_zone(), etendue=e, boite=cfg.BOITES[cfg.ZONE] or None,
                  taille_marqueur=taille, produit=cfg.PRODUIT, version=cfg.VERSION, unite=cfg.UNITE, alpha=cfg.ALPHA, z_seuil=cfg.Z_SEUIL,
                  fin_pre=cfg.FIN_PRE[:4], genere_le=date.today().isoformat(),
                  debut=ctx.etiq[0], fin=ctx.etiq[-1]),
        etiq=ctx.etiq, hunga=ctx.hunga, npre=npre,
        lat=liste(carte_ref.lat.values, 2), lon=liste(carte_ref.lon.values, 2),
        niveaux=sorties, niveau_defaut=niveaux.index(float(ds.plev.sel(plev=ctx.niv, method="nearest"))),
        profils=profils,
    )
    sortie = cfg.SITE / "data" / "donnees.js"
    sortie.write_text("window.DONNEES = " + json.dumps(donnees, ensure_ascii=False, separators=(",", ":")) + ";\n",
                      encoding="utf-8")
    print(f"{sortie}  ({sortie.stat().st_size / 1e6:.2f} Mo) – niveaux : {niveaux}")

    for f in cfg.SORTIE_ZONE.glob("*.csv"):                     # CSV téléchargeables depuis le site
        shutil.copy(f, cfg.SITE / "data" / "csv" / f.name)
    print("CSV copiés dans", cfg.SITE / "data" / "csv")


main()

## Télécharger les résultats
Récupère sur votre ordinateur les graphiques et CSV du Maroc, et `donnees.js` à placer dans `site/data/` du projet VS Code.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("resultats_maroc", "zip", "resultats/maroc")
files.download("resultats_maroc.zip")
files.download("site/data/donnees.js")